In [14]:
from firecrawl import FirecrawlApp
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

app = FirecrawlApp(api_key=os.getenv("FIRECRAWL_API_KEY"))
seller = 'https://whatfix.com/'


In [ ]:

def get_landing_page_data(url, prompt, enable_web_search=True):
    response = app.extract([url], {
        'prompt': prompt,
        'enableWebSearch': enable_web_search,
    })
    return response['data']
prompt = """
Extract all relevant information to an account executive about the product he or she is selling from the landing page. 
Extract details on the industry, product details, features it has, problems and pains it solves for its customers and any more details about its customers.
Also capture messaging on how the product presents itself to buyers.
"""
seller_landing_page_data = get_landing_page_data(seller)

ValueError: Either prompt or schema is required

In [ ]:


def get_seller_links(seller, only_top_level=True):
    """
    Get all links for a given seller.
    :param seller: The seller URL.
    :return: List of links.
    """
    # Fetch the links using the map_url method
    if seller[-1] == '/':
        seller = seller[:-1]
        
    all_links = app.map_url(seller)
    links = [link for link in all_links['links'] if link.startswith("https://") and seller.replace("https://", "") in link and link != seller]
    top_level_links = list()
    for link in links:
        if only_top_level and link.startswith(f'{seller}') and link.replace(f"{seller}", "").count('/') == 1:
            top_level_links.append(link)
    return top_level_links

top_level_relevant_links = get_seller_links(seller)
top_level_relevant_links

['https://whatfix.com/newsroom',
 'https://whatfix.com/blog',
 'https://whatfix.com/customers',
 'https://whatfix.com/solutions',
 'https://whatfix.com/compare',
 'https://whatfix.com/cookie-policy',
 'https://whatfix.com/request-trial',
 'https://whatfix.com/request-demo',
 'https://whatfix.com/product-tour',
 'https://whatfix.com/software-clicks',
 'https://whatfix.com/request-quote',
 'https://whatfix.com/why-whatfix',
 'https://whatfix.com/about-us',
 'https://whatfix.com/digital-transformation',
 'https://whatfix.com/security-framework-policy',
 'https://whatfix.com/intercom-alternatives-competitors',
 'https://whatfix.com/clicklearn-alternatives-competitors',
 'https://whatfix.com/assima-alternatives-competitors',
 'https://whatfix.com/apty-alternatives-competitors',
 'https://whatfix.com/tango-alternatives-competitors',
 'https://whatfix.com/mixpanel-alternatives-competitors',
 'https://whatfix.com/uperform-alternatives-competitors',
 'https://whatfix.com/performance-support-sys

In [10]:
from crewai import Agent, Task
from pydantic import BaseModel, Field
from echo.echo_agent import EchoAgent
from echo.utils import format_response, get_crew_llm


def extract_instruction_information_from_webpage_content(
    links_str: str,
    categories_str: str,
):
    
    class CategoryAssignment(BaseModel):
        url: str = Field(description="The URL of the webpage")
        category: str = Field(description="The category of the webpage")
    
    class AssignmentResponse(BaseModel):
        assignments : list[CategoryAssignment] = Field(description="The list of assignments")
        
    agent = Agent(
        role="Website Content Extraction Expert",
        goal="Extract out the information according to the instruction provided",
        backstory="You are an expert in extracting out the information according to the instruction provided",
        llm=get_crew_llm(),
    )
    
    task = Task(
        name="Extracting Information",
        description=(
            "Given a set of links and their title and description assign each link to one of the following categories that might be relevant for an Account Executive to understand the company's offerings and services.\n"
            "A link is relevant if it can belong to one of the categories below: \n"
            "If a link is not relevant to any of the categories, assign it to the 'Other' category.\n"
            "The categories are as follows:\n"
            "{categories}\n\n"
            
            "The links are as follows:\n"
            "{links}\n"
                        
            "Based on this information, extract out the information according to the instruction provided.\n"
            "You need to extract the relevant information that satisfies the instruction provided.\n"
            "You also need to provide a boolean value that indicates whether the instruction was satisfied or not.\n"
        ),
        expected_output=(
            "The response should conform to the provided schema."
            "You need to extract the following information in the following pydantic structure -\n"
            "{pydantic_structure}\n"
        ),
        output_pydantic=AssignmentResponse,
        agent=agent,
    )

    crew = EchoAgent(
        agents=[agent], 
        tasks=[task]
    )

    inputs = {
        "categories": categories_str,
        "links": links_str,
    }
    print("Inputs: ", inputs)

    response = crew.kickoff(inputs=inputs)
    response = format_response(response)
    return response


In [11]:
from echo.data.utils import get_relevant_link_categories

categories = get_relevant_link_categories()
category_prompts = {
    category: (
        f"Task: {r['Prompt']}\n"
        f"Purpose: {r['Purpose']}\n"
        f"Examples: {r['Examples']}\n"
    )
    for category, r in categories.items()
}

In [ ]:
import concurrent.futures
from typing import List
from langchain.document_loaders import SeleniumURLLoader



def extract_relevant_urls(links: List[str], categories, BATCH_SIZE: int = 50) -> List[dict]:
    prompts_args = list()
    final_docs = SeleniumURLLoader(urls=links).load()
    
    for i in range(0, len(final_docs), BATCH_SIZE):
        batch = final_docs[i:i + BATCH_SIZE]
        links_str = "\n".join([f"URL:{doc.metadata['source']}\n{doc.metadata['title']}: {doc.metadata['description']}" for doc in batch])
        categories_str = "\n".join(
            f"{i+1}. Category: {category}\nPurpose: {r['Purpose']}\nExamples:{r['Examples'].replace("\n", ', ')}\n" for i, (category, r) in enumerate(categories.items())
        )
        prompts_args.append((links_str, categories_str))
    
    extracted_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = {
            executor.submit(extract_instruction_information_from_webpage_content, links_str, categories_str): idx
            for idx, (links_str, categories_str) in enumerate(prompts_args)
        }
        for future in concurrent.futures.as_completed(results):
            idx = results[future]
            try:
                extracted_results.append(future.result())
            except Exception as exc:
                print(f"Prompt {idx} generated an exception: {exc}")

    relevant_links = list()
    for response in extracted_results:    
        if not isinstance(response, str):
            assignments = response['assignments']    
            for assignment in assignments:
                if assignment['category'].lower() != 'other':
                    relevant_links.append(assignment)
    
    links_by_category = {}
    for link in relevant_links:
        category = link['category']
        if category not in links_by_category:
            links_by_category[category] = {'prompt': categories[category]['Prompt'], 'links': []}
        links_by_category[category]['links'].append(link['url'])
    
    return links_by_category

final_relevant_links_by_category = extract_relevant_urls(top_level_relevant_links, categories=categories)

In [21]:
for k, v in final_relevant_links_by_category.items():
    print(k, len(v['links']))

Hero Headline & Subhead 3
Blog & Thought Leadership 1
Customer Logos 1
Features & Capabilities 15
CTA / Conversion Copy 2
Use Case Pages 3


In [25]:
final_relevant_links_by_category

{'Hero Headline & Subhead': {'prompt': 'Extract the main headline and subheadline. Identify the core product promise, target users, and market positioning in 1–2 sentences.',
  'links': ['https://whatfix.com/ai',
   'https://whatfix.com/software-clicks',
   'https://whatfix.com/why-whatfix']},
 'Blog & Thought Leadership': {'prompt': 'Scan latest 3–5 blog titles and summaries. Extract the core themes, company POV, and any strategic messaging tone relevant to outbound narrative.',
  'links': ['https://whatfix.com/blog']},
 'Customer Logos': {'prompt': 'Extract the names of all customer logos shown. If possible, infer industry and company size from context or URL.',
  'links': ['https://whatfix.com/customers']},
 'Features & Capabilities': {'prompt': 'List all product features, capabilities, and integrations mentioned. Summarize each feature in a sentence, and categorize by theme (e.g. AI, integrations, UI).',
  'links': ['https://whatfix.com/solutions',
   'https://whatfix.com/compare',

In [ ]:
from tqdm.auto import tqdm

LINK_BATCH_SIZE = 10

def get_data_from_links_by_category(links_by_category):
    category_data = dict()
    for category, data in tqdm(links_by_category.items()):
        links = data['links']
        prompt = data['prompt']
        link_batches = [links[i:i + LINK_BATCH_SIZE] for i in range(0, len(links), LINK_BATCH_SIZE)]
        print("Number of batches: ", len(link_batches))
        responses = [None]*len(link_batches)
        with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
            futures = {
                executor.submit(app.extract, link_batch, {
                    'prompt': prompt,
                    'enableWebSearch': True,
                }): idx
                for idx, link_batch in enumerate(link_batches)
            }
            for future in concurrent.futures.as_completed(futures):
                idx = futures[future]
                try:
                    responses[idx] = future.result()
                except Exception as exc:
                    print(f"Prompt {idx} generated an exception: {exc}")
        responses = [r for r in responses if r is not None]
        
        print(f"Processed {category} - {len(responses)} responses")
            
        category_data[category] = responses
    return category_data

category_data = get_data_from_links_by_category(final_relevant_links_by_category)

In [ ]:
import json
with open('category_data_single.json', 'w') as f:
    json.dump(category_data, f)

In [38]:
from echo.utils import json_to_markdown

for k, v in category_data.items():
    print(k)
    for i in v:
        print(json_to_markdown(i['data']))
        print("-"*50)
    


Hero Headline & Subhead
# Sub Headline
Whatfix, a global leader in digital adoption platforms (DAPs), today announced that it has been named a Leader in The Forrester Wave™: Digital Adoption Platforms, Q4 2024 report.
# Target Users
The platform is designed for enterprises and organizations looking to improve user adoption of their software applications, including Fortune 500 companies.
# Main Headline
Whatfix Named a Leader in The Forrester Wave™: Digital Adoption Platforms, Q4 2024
# Market Positioning
Whatfix is positioned as a leader in the digital adoption space, recognized for its innovative features, ease of implementation, and strong customer support.
# Core Product Promise
Whatfix's Digital Adoption Platform (DAP) provides contextual in-app guidance and moment-of-need support to enhance user engagement and drive business outcomes.

--------------------------------------------------
Blog & Thought Leadership
# Blogs
   - ### Title
Whatfix Recognized as a Digital Adoption Platfo

In [34]:
len(category_data['Features & Capabilities'])

2